## Here we plot some benchmark results. The benchmark were performed on Google Colab CPU or GPU (free tier) in order to be reproducible.

### Useful functions

In [1]:
import time
import jax
jax.config.update("jax_enable_x64", True)
jax.config.update('jax_platform_name', 'gpu') # put 'gpu' here if you have one
import jax.numpy as jnp

def run_benchmark(func, func_args, func_kwargs, repetition=10):
  # warmuo
  func(*func_args, **func_kwargs)
  times = []
  for _ in range(repetition):
    t0 = time.time()
    res = func(*func_args, **func_kwargs).block_until_ready()
    tf = time.time()
    times.append(tf-t0)
  return jnp.mean(jnp.asarray(times)), jnp.std(jnp.asarray(times))

def jax_gkde_2d(x_grid, y_grid, data, weights=None, bw_method=None):
  weights = weights / jnp.sum(weights) if weights is not None else None

  # Create KDE estimator
  kde = jax.scipy.stats.gaussian_kde(data.T, bw_method=bw_method, weights=weights)

  # Evaluate on grid
  X, Y = jnp.meshgrid(x_grid, y_grid, indexing='ij')
  positions = jnp.vstack([X.ravel(), Y.ravel()])
  density = kde(positions).reshape(X.shape)
  return density


def generate_data_2d(key, n_samples=10_000):
  keys = jax.random.split(key, 5)
  data = []
  data.append(jax.random.multivariate_normal(keys[0],
            mean=jnp.array([1.0, 1.0]),
            cov=jnp.array([[0.7, 0.3], [0.3, 0.5]]),
            shape=(n_samples//2,)))
  data.append(jax.random.multivariate_normal(keys[1],
            mean=jnp.array([-1.0, -1.0]),
            cov=jnp.array([[0.5, -0.2], [-0.2, 0.8]]),
            shape=(n_samples//2,)))

  return jnp.concatenate(data)

def generate_3d_data(key, n_samples=5000):
  keys = jax.random.split(key, 4)
  data = []

  # Component 1
  data.append(jax.random.multivariate_normal(
      keys[0],
      mean=jnp.array([1.0, 1.0, 0.5]),
      cov=jnp.array([[0.7, 0.3, 0.1], [0.3, 0.5, 0.2], [0.1, 0.2, 0.6]]),
      shape=(n_samples//3,)
  ))

  # Component 2
  data.append(jax.random.multivariate_normal(
      keys[1],
      mean=jnp.array([-1.0, -1.0, -0.5]),
      cov=jnp.array([[0.5, -0.2, 0.0], [-0.2, 0.8, -0.1], [0.0, -0.1, 0.4]]),
      shape=(n_samples//3,)
  ))

  # Component 3
  data.append(jax.random.multivariate_normal(
      keys[2],
      mean=jnp.array([0.0, 1.5, -1.0]),
      cov=jnp.array([[0.3, 0.0, 0.0], [0.0, 0.3, 0.0], [0.0, 0.0, 0.9]]),
      shape=(n_samples//3,)
  ))

  return jnp.concatenate(data)

def jax_gkde_3d(x_grid, y_grid, z_grid, data, weights=None, bw_method=None):
    weights = weights / jnp.sum(weights) if weights is not None else None
    kde = jax.scipy.stats.gaussian_kde(data.T, bw_method=bw_method, weights=weights)

    X, Y, Z = jnp.meshgrid(x_grid, y_grid, z_grid, indexing='ij')
    positions = jnp.vstack([X.ravel(), Y.ravel(), Z.ravel()])
    return kde(positions).reshape(X.shape)

# 2-dimensional case

## 1. varying points resolution

In [ ]:
from collections import defaultdict
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)
dir_parent    = "/content/gdrive/MyDrive/KDExpress/src/"
import sys
sys.path.append(dir_parent)
from KDExpress import fft_kde2d, silverman_bw2d, build_hist_edges


key = jax.random.PRNGKey(42)
data_2d = generate_data_2d(key, n_samples=5_000)
weights = data_2d[:,0]**2
x_min, x_max = data_2d[:,0].min()-0.5, data_2d[:,0].max()+0.5
y_min, y_max = data_2d[:,1].min()-0.5, data_2d[:,1].max()+0.5
points_x = jnp.linspace(x_min, x_max, 100)
points_y = jnp.linspace(y_min, y_max, 100)
be = [build_hist_edges(p) for p in [points_x, points_y]]

fft_kde_2d_args = [points_x, points_y, data_2d]
fft_kde_2d_kwargs = {'weights': weights}
fft_kde_2d_kwargs_with_be = {'weights': weights, 'bin_edges':be}

jax_kde_args = [points_x, points_y, data_2d]
jax_kde_kwargs = {'weights': weights, 'bw_method':silverman_bw2d(data_2d)[0]}

methods = {
  'fft_kde': (fft_kde2d, fft_kde_2d_args, fft_kde_2d_kwargs),
  'fft_kde_with_be': (fft_kde2d, fft_kde_2d_args, fft_kde_2d_kwargs_with_be),
  'jax_kde': (jax_gkde_2d, jax_kde_args, jax_kde_kwargs)
}

def compute_benchmark_point_res(points_res):
  mean_times = defaultdict(list)
  for res in points_res:
    points_x = jnp.linspace(x_min, x_max, res)
    points_y = jnp.linspace(y_min, y_max, res)
    be = [build_hist_edges(p) for p in [points_x, points_y]]
    fft_kde_2d_args[0] = points_x
    fft_kde_2d_args[1] = points_y

    jax_kde_args[0] = points_x
    jax_kde_args[1] = points_y

    fft_kde_2d_kwargs_with_be['bin_edges'] = be

    print(f"\nResolution: {res} points")
    for method_name, (func, args, kwargs) in methods.items():
      mean, std = run_benchmark(func, args, kwargs)
      mean_times[method_name].append(float(mean))
      print(f"{method_name}: {mean:.5f} ± {std:.5f} s")
  return dict(mean_times)


points_res = [50,100,150,200, 250]
mean_time_point_res = compute_benchmark_point_res(points_res)

## 2. varying data resolution

In [ ]:
key = jax.random.PRNGKey(42)
data_2d = generate_data_2d(key, n_samples=5_000)
weights = data_2d[:,0]**2
x_min, x_max = data_2d[:,0].min()-0.5, data_2d[:,0].max()+0.5
y_min, y_max = data_2d[:,1].min()-0.5, data_2d[:,1].max()+0.5
points_x = jnp.linspace(x_min, x_max, 100)
points_y = jnp.linspace(y_min, y_max, 100)
be = [build_hist_edges(p) for p in [points_x, points_y]]

fft_kde_2d_args = [points_x, points_y, data_2d]
fft_kde_2d_kwargs = {'weights': weights}
fft_kde_2d_kwargs_with_be = {'weights': weights, 'bin_edges':be}

jax_kde_args = [points_x, points_y, data_2d]
jax_kde_kwargs = {'weights': weights, 'bw_method':silverman_bw2d(data_2d)[0]}

methods = {
  'fft_kde': (fft_kde2d, fft_kde_2d_args, fft_kde_2d_kwargs),
  'fft_kde_with_be': (fft_kde2d, fft_kde_2d_args, fft_kde_2d_kwargs_with_be),
  'jax_kde': (jax_gkde_2d, jax_kde_args, jax_kde_kwargs)
}

def compute_benchmark_data_res(data_res):
  mean_times = defaultdict(list)
  for res in data_res:
    key = jax.random.PRNGKey(42)
    data_2d = generate_data_2d(key, n_samples=res)
    weights = data_2d[:,0]**2
    points_x = jnp.linspace(x_min, x_max, 100)
    points_y = jnp.linspace(y_min, y_max, 100)
    be = [build_hist_edges(p) for p in [points_x, points_y]]
    fft_kde_2d_args[0] = points_x
    fft_kde_2d_args[1] = points_y
    fft_kde_2d_args[2] = data_2d
    fft_kde_2d_kwargs['weights'] = weights
    fft_kde_2d_kwargs_with_be['weights'] = weights

    jax_kde_args[0] = points_x
    jax_kde_args[1] = points_y
    jax_kde_args[2] = data_2d
    jax_kde_kwargs['weights'] = weights

    fft_kde_2d_kwargs_with_be['bin_edges'] = be

    print(f"\nResolution: {res} data")
    for method_name, (func, args, kwargs) in methods.items():
      mean, std = run_benchmark(func, args, kwargs)
      mean_times[method_name].append(float(mean))
      print(f"{method_name}: {mean:.5f} ± {std:.5f} s")
  return dict(mean_times)


data_res = [2500, 5000, 7500, 10000, 12500]
mean_time_data_res = compute_benchmark_data_res(data_res)

## 3. Plots for 2-D

In [ ]:
import matplotlib.pyplot as plt

# Create a figure with two subplots side by side with shared y-axis
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

# First subplot - Points Resolution
for method in methods:
    ax1.plot(points_res, mean_time_point_res[method], marker='o', label=method)
ax1.set_xlabel('Points Resolution [Data = (5_000,2)]')
ax1.set_ylabel('Mean Execution Time (s)')
ax1.set_yscale('log')
ax1.set_title('CPU Times')
ax1.legend()
ax1.grid(True)

# Second subplot - Data Resolution
for method in methods:
    ax2.plot(data_res, mean_time_data_res[method], marker='o', label=method)
ax2.set_xlabel('Data Resolution [Points = (100,2)]')
ax2.set_title('CPU Times')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig(dir_parent+'../examples/cpu_bench_2d.png', dpi=600)
plt.show()

# 3D case

## 1. varying point resolution

In [ ]:
from collections import defaultdict
import os,sys
from KDExpress import fft_kde3d, silverman_bw3d, build_hist_edges

key = jax.random.PRNGKey(42)
data_3d = generate_3d_data(key, n_samples=2500)
weights = (data_3d**2)[:,0]

dim =  50
bounds = [(data_3d[:,i].min()-0.5, data_3d[:,i].max()+0.5) for i in range(3)]
grids = [jnp.linspace(b[0], b[1], dim) for b in bounds]
points_x = grids[0]
points_y = grids[1]
points_z = grids[2]
px, py, pz = jnp.meshgrid(points_x, points_y, points_z, indexing='ij')
be = [build_hist_edges(points) for points in [points_x, points_y, points_z]]

fft_kde_3d_args = [points_x, points_y, points_z, data_3d]
fft_kde_3d_kwargs = {'weights': weights}
fft_kde_3d_kwargs_with_be = {'weights': weights, 'bin_edges':be}

jax_kde_args = [points_x, points_y, points_z, data_3d]
jax_kde_kwargs = {'weights': weights, 'bw_method':silverman_bw3d(data_3d)[0]}

methods = {
  'fft_kde': (fft_kde3d, fft_kde_3d_args, fft_kde_3d_kwargs),
  'fft_kde_with_be': (fft_kde3d, fft_kde_3d_args, fft_kde_3d_kwargs_with_be),
  'jax_kde': (jax_gkde_3d, jax_kde_args, jax_kde_kwargs)
}

def compute_benchmark_point_res(points_res):
  mean_times = defaultdict(list)
  for res in points_res:
    grids = [jnp.linspace(b[0], b[1], res) for b in bounds]
    points_x = grids[0]
    points_y = grids[1]
    points_z = grids[2]
    px, py, pz = jnp.meshgrid(points_x, points_y, points_z, indexing='ij')
    be = [build_hist_edges(points) for points in [points_x, points_y, points_z]]

    fft_kde_3d_args[0] = points_x
    fft_kde_3d_args[1] = points_y
    fft_kde_3d_args[2] = points_x

    jax_kde_args[0] = points_x
    jax_kde_args[1] = points_y
    jax_kde_args[2] = points_z

    fft_kde_3d_kwargs_with_be['bin_edges'] = be

    print(f"\nResolution: {res} points")
    for method_name, (func, args, kwargs) in methods.items():
      mean, std = run_benchmark(func, args, kwargs)
      mean_times[method_name].append(float(mean))
      print(f"{method_name}: {mean:.5f} ± {std:.5f} s")
  return dict(mean_times)


points_res = [15, 30, 45, 60, 75]
mean_time_point_res = compute_benchmark_point_res(points_res)

## 2. varying data resolution

In [ ]:
key = jax.random.PRNGKey(42)
data_3d = generate_3d_data(key, n_samples=2500)
weights = (data_3d**2)[:,0]

bounds = [(data_3d[:,i].min()-0.5, data_3d[:,i].max()+0.5) for i in range(3)]
grids = [jnp.linspace(b[0], b[1], 25) for b in bounds]
points_x = grids[0]
points_y = grids[1]
points_z = grids[2]
px, py, pz = jnp.meshgrid(points_x, points_y, points_z, indexing='ij')
be = [build_hist_edges(points) for points in [points_x, points_y, points_z]]

fft_kde_3d_args = [points_x, points_y, points_z, data_3d]
fft_kde_3d_kwargs = {'weights': weights}
fft_kde_3d_kwargs_with_be = {'weights': weights, 'bin_edges':be}

jax_kde_args = [points_x, points_y, points_z, data_3d]
jax_kde_kwargs = {'weights': weights, 'bw_method':silverman_bw3d(data_3d)[0]}

methods = {
  'fft_kde': (fft_kde3d, fft_kde_3d_args, fft_kde_3d_kwargs),
  'fft_kde_with_be': (fft_kde3d, fft_kde_3d_args, fft_kde_3d_kwargs_with_be),
  'jax_kde': (jax_gkde_3d, jax_kde_args, jax_kde_kwargs)
}

def compute_benchmark_point_res(data_res):
  mean_times = defaultdict(list)
  for res in data_res:
    key = jax.random.PRNGKey(42)
    data_3d = generate_3d_data(key, n_samples=res)
    weights = (data_3d**2)[:,0]
    bounds = [(data_3d[:,i].min()-0.5, data_3d[:,i].max()+0.5) for i in range(3)]
    grids = [jnp.linspace(b[0], b[1], 25) for b in bounds]
    points_x = grids[0]
    points_y = grids[1]
    points_z = grids[2]
    px, py, pz = jnp.meshgrid(points_x, points_y, points_z, indexing='ij')
    be = [build_hist_edges(points) for points in [points_x, points_y, points_z]]

    fft_kde_3d_args[0] = points_x
    fft_kde_3d_args[1] = points_y
    fft_kde_3d_args[2] = points_x
    fft_kde_3d_args[3] = data_3d

    jax_kde_args[0] = points_x
    jax_kde_args[1] = points_y
    jax_kde_args[2] = points_z
    jax_kde_args[3] = data_3d

    fft_kde_3d_kwargs['weights'] = weights
    fft_kde_3d_kwargs_with_be['weights'] = weights
    jax_kde_kwargs['weights'] = weights
    fft_kde_3d_kwargs_with_be['bin_edges'] = be

    print(f"\nResolution: {res} points")
    for method_name, (func, args, kwargs) in methods.items():
      mean, std = run_benchmark(func, args, kwargs)
      mean_times[method_name].append(float(mean))
      print(f"{method_name}: {mean:.5f} ± {std:.5f} s")
  return dict(mean_times)


data_res = [3000, 4000, 5000, 6000, 7000]
mean_time_data_res = compute_benchmark_point_res(data_res)

## 3. 3d plots

In [ ]:
import matplotlib.pyplot as plt

# Create a figure with two subplots side by side with shared y-axis
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

# First subplot - Points Resolution
for method in methods:
    ax1.plot(points_res, mean_time_point_res[method], marker='o', label=method)
ax1.set_xlabel('Points Resolution [Data = (2500,3)]')
ax1.set_ylabel('Mean Execution Time (s)')
ax1.set_yscale('log')
ax1.set_title('GPU Times')
ax1.legend()
ax1.grid(True)

# Second subplot - Data Resolution
for method in methods:
    ax2.plot(data_res, mean_time_data_res[method], marker='o', label=method)
ax2.set_xlabel('Data Resolution [Points = (25,3)]')
ax2.set_title('GPU Times')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig(dir_parent+'../examples/cpu_bench_3d.png', dpi=600)
plt.show()